# 04 — Eight-view reconstruction-input EDA

This notebook reads the **3DGS inputs produced by notebook 03** and compares exactly eight selected training views per retained scene. It does not resize or overwrite reconstruction images. A temporary 256×256 representation is used only to make descriptive statistics fast and comparable; this analysis resize is never saved as model input.

3DRealCar held-out views remain available in the method manifest for later novel-view evaluation, but they are excluded from this input EDA. IndustrialInventory has only its eight reconstruction inputs.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
import torch

CPU_ONLY_NOTEBOOK = True
GPU_ATTACHED = torch.cuda.is_available()

print("Runtime check")
print("-" * 50)
print("GPU attached:", GPU_ATTACHED)

if GPU_ATTACHED:
    print("GPU:", torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU memory: {free_bytes / 1024**3:.2f} GB free / {total_bytes / 1024**3:.2f} GB total")

if CPU_ONLY_NOTEBOOK and GPU_ATTACHED:
    raise RuntimeError(
        "This notebook is CPU-only. To conserve Colab GPU availability, "
        "change the Colab hardware accelerator to None, reconnect the CPU "
        "kernel in VS Code, and run the notebook again."
    )

print("Correct CPU runtime: continue with the notebook.")


In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))

# Change only this value if the Drive project is moved.
PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "ITU" / "3D" / "Thesis"

def find_unique_dir(names, search_roots):
    direct = [root / name for root in search_roots for name in names]
    matches = [p for p in direct if p.is_dir()]
    if not matches:
        for root in search_roots:
            if root.is_dir():
                matches.extend(p for p in root.rglob("*") if p.is_dir() and p.name.casefold() in {n.casefold() for n in names})
    unique = list(dict.fromkeys(p.resolve() for p in matches))
    if len(unique) != 1:
        raise FileNotFoundError(f"Expected one of {names}; found {len(unique)}: {unique}")
    return unique[0]

SEARCH_ROOTS = [PROJECT_ROOT / "data", PROJECT_ROOT]
INDUSTRIAL_ROOT = find_unique_dir(["IndustrialInventory"], SEARCH_ROOTS)
HQ200_ROOT = find_unique_dir(["3DrealCarHQ200", "HQ200"], SEARCH_ROOTS)

# If HQ200 is a wrapper folder, descend to the folder containing capture scenes.
if (HQ200_ROOT / "3DrealCarHQ200").is_dir():
    HQ200_ROOT = HQ200_ROOT / "3DrealCarHQ200"

print("Project:   ", PROJECT_ROOT)
print("Industrial:", INDUSTRIAL_ROOT)
print("3DRealCar: ", HQ200_ROOT)


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "https://github.com/katlit/Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"

if not (CODE_ROOT / "code" / "src").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)

CODE_PACKAGE_ROOT = CODE_ROOT / "code"
if str(CODE_PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_PACKAGE_ROOT))

print("Reusable code:", CODE_ROOT / "code" / "src")


In [ ]:
from src.eda_utils import masked_image_statistics
from src.image_preprocessing import VIEW_LABELS

METHOD_ROOT = PROJECT_ROOT / "data_processed" / "method_inputs"
MANIFEST_PATH = METHOD_ROOT / "manifest.csv"
EDA_ROOT = PROJECT_ROOT / "splits" / "sparse8"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Run notebook 03 with RUN_EXPORT=True first: {MANIFEST_PATH}")

method_manifest = pd.read_csv(MANIFEST_PATH)
eda_inputs = method_manifest.query("method == '3dgs' and split == 'train'").copy()
counts = eda_inputs.groupby(["dataset", "scene"]).size()
assert counts.eq(8).all(), "EDA requires exactly eight selected inputs for every scene."
print(f"Scenes: {len(counts)} | reconstruction inputs: {len(eda_inputs)}")
display(counts.rename("views").to_frame())


## Masked statistics

RGB and masks are sampled temporarily at 256×256 for computationally light dataset characterization. Brightness, contrast, and the gradient-based sharpness proxy are computed only inside the foreground mask. These are descriptive proxies, not perceptual-quality scores and not reconstruction preprocessing.


In [ ]:
statistics = []
for row in tqdm(eda_inputs.itertuples(index=False), total=len(eda_inputs), desc="Analysis-only 256px statistics"):
    statistics.append(masked_image_statistics(row.method_image, row.method_mask, sample_size=(256, 256)))
eda = pd.concat([eda_inputs.reset_index(drop=True), pd.DataFrame(statistics)], axis=1)
EDA_ROOT.mkdir(parents=True, exist_ok=True)
eda.to_csv(EDA_ROOT / "image_statistics.csv", index=False)

comparison = eda.groupby("dataset").agg(
    scenes=("scene", "nunique"), images=("method_image", "size"),
    native_width_mean=("method_width", "mean"), native_height_mean=("method_height", "mean"),
    foreground_fraction_mean=("foreground_fraction", "mean"),
    brightness_mean=("brightness", "mean"), brightness_std=("brightness", "std"),
    contrast_mean=("contrast", "mean"), sharpness_mean=("sharpness_proxy", "mean"),
).round(2)
display(comparison)

scene_comparison = eda.groupby(["dataset", "scene"]).agg(
    input_views=("method_image", "size"),
    foreground_fraction_mean=("foreground_fraction", "mean"),
    brightness_mean=("brightness", "mean"),
    contrast_mean=("contrast", "mean"),
    sharpness_mean=("sharpness_proxy", "mean"),
).round(2).reset_index()
assert scene_comparison.input_views.eq(8).all()
display(scene_comparison)
scene_comparison.to_csv(EDA_ROOT / "scene_statistics_8views.csv", index=False)


In [ ]:
metrics = ["foreground_fraction", "brightness", "contrast", "sharpness_proxy"]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, metric in zip(axes, metrics):
    for dataset, group in eda.groupby("dataset"):
        ax.hist(group[metric], bins=20, density=True, alpha=0.45, label=dataset)
    ax.set_title(metric.replace("_", " ").title())
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
def show_all_selected(frame, dataset):
    part = frame[frame.dataset == dataset]
    scenes = sorted(part.scene.unique())
    fig, axes = plt.subplots(len(scenes), 8, figsize=(24, 3 * len(scenes)), squeeze=False)
    for r, scene in enumerate(scenes):
        group = part[part.scene == scene].sort_values(["view_order", "source"])
        for c, (_, row) in enumerate(group.iterrows()):
            with Image.open(row.method_image) as image:
                axes[r, c].imshow(image.convert("RGB"))
            axes[r, c].axis("off")
            if r == 0:
                axes[r, c].set_title(VIEW_LABELS[c])
        axes[r, 0].set_ylabel(scene, rotation=0, ha="right", labelpad=90, fontsize=8)
    fig.suptitle(f"{dataset}: eight annotated reconstruction inputs per scene")
    plt.tight_layout()
    plt.show()

show_all_selected(eda, "3DRealCar")
show_all_selected(eda, "IndustrialInventory")


## Interpretation

Both datasets contribute eight training inputs per retained car; the datasets differ in their number of car scenes. Therefore raw image counts should not be interpreted as balanced sampling. Use 3DRealCar non-training frames only for held-out rendering metrics. Industrial training-view agreement is not unseen-view performance.
